<a href="https://colab.research.google.com/github/jiuwong/sfu_AppliedAI_DataAnalytics/blob/main/8_1_zero_shot_classification_for_business_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://sfudial.ca/wp-content/uploads/SFU-DIAL-Logo.png" width=40%>&nbsp;&nbsp;&nbsp;&nbsp;<img src="https://www.sfu.ca/content/dam/sfu/images/brand_extension/SFU-Big-Data_Logo.png" width=40%>                                                                                          

# Lab 8: Zero-Shot Classification for Business Analytics

**Advanced AI Techniques for Business Intelligence**

## What is Zero-Shot Classification?

Zero-shot classification allows you to classify text into **any categories you define** without requiring training data. The AI model uses its pre-trained understanding of language to match your text to your custom categories.

### What Zero-Shot Classification Can Do

- ✅ **Classify text into custom categories** - Define any categories that make sense for your business
- ✅ **Work without training data** - No need to collect and label examples
- ✅ **Handle unstructured text** - Works with messy, real-world business data
- ✅ **Provide confidence scores** - See how certain the AI is about each classification
- ✅ **Support multiple languages** - Works with text in various languages (with appropriate models)
- ✅ **Flexible category changes** - Modify categories anytime without retraining

### What Zero-Shot Classification Cannot Do

- ❌ **Highly specialized domain knowledge** - May struggle with very technical or niche terminology
- ❌ **Extremely fine-grained distinctions** - Better at broader categories than subtle differences
- ❌ **Context beyond the text** - Only analyzes the provided text, not external context
- ❌ **Multi-label classification** - Typically assigns one primary category per text
- ❌ **Real-time streaming** - Requires processing time (though fast for single texts)
- ❌ **Guaranteed accuracy** - Confidence scores indicate uncertainty, but errors can occur

### Good For These Scenarios

- ✅ **Customer support ticket routing** - Classify tickets into support categories
- ✅ **Content categorization** - Organize articles, reviews, or documents
- ✅ **Risk assessment** - Identify potential issues in business reports
- ✅ **Sentiment-based classification** - Categorize feedback as positive, negative, neutral
- ✅ **Quick prototyping** - Test classification ideas without building training datasets
- ✅ **Dynamic category needs** - When categories change frequently
- ✅ **Multi-language support** - Classify text in different languages
- ✅ **Low-resource scenarios** - When you don't have labeled training data

### Bad For These Scenarios

- ❌ **Highly regulated domains** - Where classification accuracy must be guaranteed (e.g., medical diagnosis)
- ❌ **Very domain-specific terminology** - Technical jargon not in the model's training data
- ❌ **Real-time high-volume processing** - When you need to classify millions of texts per second
- ❌ **Multi-label requirements** - When texts must belong to multiple categories simultaneously
- ❌ **Extremely nuanced distinctions** - When categories are very similar (e.g., "slightly positive" vs "moderately positive")
- ❌ **Context-dependent classification** - When classification depends on external information not in the text
- ❌ **Production systems requiring 99%+ accuracy** - When errors have severe consequences

In [ ]:
from transformers import pipeline
import json
import pandas as pd

## Step 1: Initialize the Classifier

Choose between two models:
- **BART-Large**: Higher accuracy (~1.6GB), slower processing
- **BART-Base**: Faster processing (~600MB), slightly lower accuracy

Edit the code below to switch models by commenting/uncommenting the appropriate lines.

In [ ]:
classifier = pipeline("zero-shot-classification",
                     model="facebook/bart-large-mnli")

# Uncomment the line below and comment the line above to use the faster model:
# classifier = pipeline("zero-shot-classification",
#                      model="facebook/bart-base-mnli")

## Step 2: Prepare Your Business Data

Add the text data you want to classify. You can include as many texts as needed in the list below.

In [ ]:
business_texts = [
    "Customer complained about slow checkout process",
    "Website traffic dropped 40% after redesign",
    "New product line showing strong review sentiment",
    "Payment processing errors increased this week",
    "User engagement metrics are trending upward"
]

## Step 3: Define Your Categories

Specify the categories you want to use for classification. Categories should be:
- Descriptive and distinct from each other
- Relevant to your business domain
- Clear enough for the AI to distinguish between them

In [ ]:
business_categories = [
    "user_experience_issue",
    "technical_bug",
    "payment_problem",
    "performance_anomaly",
    "positive_trend",
    "negative_trend"
]

## Step 4: Run Classification

The classifier will analyze each text and assign it to the most appropriate category. Each classification includes:
- **Predicted category**: The top match
- **Confidence score**: How certain the AI is (0.0 to 1.0)
  - High (>0.7): Very confident
  - Medium (0.4-0.7): Somewhat confident
  - Low (<0.4): Uncertain, may need review
- **All scores**: Confidence for every category

In [ ]:
results = []
for i, text in enumerate(business_texts):
    result = classifier(text, business_categories)

    classification = {
        "text": text,
        "predicted_category": result["labels"][0],
        "confidence": round(result["scores"][0], 3),
        "all_scores": dict(zip(result["labels"], [round(s, 3) for s in result["scores"]]))
    }

    results.append(classification)
    print(f"Text {i+1}: {text}")
    print(f"  → Predicted: {classification['predicted_category']} (confidence: {classification['confidence']})")
    print(f"  All category scores:")
    for category, score in classification['all_scores'].items():
        marker = "✓" if category == classification['predicted_category'] else " "
        print(f"    {marker} {category}: {score}")
    print()

In [ ]:
# Display results in a structured table
results_df = pd.DataFrame(results)
print("Classification Results Summary:")
print("=" * 80)

# Create a detailed DataFrame with all scores
detailed_data = []
for result in results:
    row = {"Text": result["text"], "Predicted": result["predicted_category"]}
    for category in business_categories:
        row[category] = result["all_scores"].get(category, 0.0)
    detailed_data.append(row)

detailed_df = pd.DataFrame(detailed_data)
print("\nDetailed Results (all category scores):")
print(detailed_df.to_string(index=False))
print()

# Summary table
summary_df = pd.DataFrame([
    {
        "Text": r["text"],
        "Predicted Category": r["predicted_category"],
        "Confidence": r["confidence"]
    }
    for r in results
])
print("\nSummary (predicted categories):")
print(summary_df.to_string(index=False))

In [ ]:
structured_results = {
    "classifications": results,
    "summary": {
        "total_texts": len(business_texts),
        "categories_used": business_categories,
        "model": "facebook/bart-large-mnli"
    }
}

## Try Your Own: Experiment with Different Categories

Create your own category sets for different business scenarios. The same text can be classified differently depending on which categories you use.

**Note:** The current filler categories and text below are an example of financial risk classification. Edit the `custom_categories` list to define your own categories, and edit the `custom_text` variable to test different scenarios.

In [ ]:
custom_categories = [
    "prepayment_risk",
    "default_risk",
    "market_risk",
    "operational_risk",
    "low_risk"
]

In [ ]:
custom_text = "Mortgage applications down 15% in Q2"

In [ ]:
test_result = classifier(custom_text, custom_categories)

In [ ]:
test_classification = {
    "text": custom_text,
    "predicted_category": test_result["labels"][0],
    "confidence": round(test_result["scores"][0], 3),
    "all_scores": dict(zip(test_result["labels"], [round(s, 3) for s in test_result["scores"]]))
}

print(f"Text: {test_classification['text']}")
print(f"Predicted Category: {test_classification['predicted_category']} (confidence: {test_classification['confidence']})")
print(f"\nAll category scores:")
for category, score in test_classification['all_scores'].items():
    marker = "✓" if category == test_classification['predicted_category'] else " "
    print(f"  {marker} {category}: {score}")

In [ ]:
# Display custom test result in a table
custom_df = pd.DataFrame([{
    "Text": test_classification["text"],
    "Predicted": test_classification["predicted_category"],
    **{cat: test_classification["all_scores"].get(cat, 0.0) for cat in custom_categories}
}])
print("\nCustom Test Results:")
print(custom_df.to_string(index=False))

## 🎉 LAB 8 COMPLETE!

**What You Accomplished:**
- ✅ Classified business text with zero-shot classification
- ✅ Used custom business categories without training data
- ✅ Generated structured JSON output for automation
- ✅ Experimented with different category sets

**Next Steps:**
- Apply zero-shot classification to your own business data
- Experiment with different category sets for various scenarios
- Integrate results into your business workflows
- Explore other advanced AI techniques in additional labs

# Task
Perform zero-shot classification on the first 200 samples of the test split from the 'amazon_polarity' dataset. Map numerical labels to 'negative review' and 'positive review' categories. After classification, calculate and display the accuracy, present a summary table of classification results (true label, predicted category, and confidence), and visualize the distribution of confidence scores. Finally, summarize the findings, including the achieved accuracy and observations from the confidence score visualization.

## Load and Prepare Dataset

### Subtask:
Load the 'amazon_polarity' dataset, specifically the first 200 samples from the test split. Extract the review text and map the numerical labels (0, 1) to descriptive string categories ('negative review', 'positive review') for zero-shot classification.

**Reasoning**:
The first instruction is to install the `datasets` library. This is a prerequisite for loading the dataset.

In [ ]:
!pip install datasets

**Reasoning**:
Now that the `datasets` library is installed, I will load the 'amazon_polarity' dataset, extract the first 200 samples from the test split, and map the numerical labels to descriptive string categories as per the subtask instructions.

In [ ]:
from datasets import load_dataset

# Load the 'amazon_polarity' dataset
dataset = load_dataset('amazon_polarity')

# Access the 'test' split and select the first 200 samples
test_split_200 = dataset['test'].select(range(200))

# Initialize an empty list to store processed data
processed_data = []

# Iterate through the samples to extract text and map labels
for item in test_split_200:
    text_content = item['content']
    numerical_label = item['label']

    # Map numerical label to descriptive string categories
    true_label = 'positive review' if numerical_label == 1 else 'negative review'

    processed_data.append({"text": text_content, "true_label": true_label})

print(f"Loaded and processed {len(processed_data)} samples.")
print("First 3 processed samples:")
for i in range(min(3, len(processed_data))):
    print(processed_data[i])

## Perform Zero-Shot Classification

### Subtask:
Iterate through the 200 prepared samples. For each sample, apply the pre-initialized zero-shot classifier using the categories 'positive review' and 'negative review'. Store the original text, true label, predicted category, and confidence score.

**Reasoning**:
I need to perform the zero-shot classification as instructed. This involves defining the sentiment categories, iterating through the prepared dataset, applying the classifier, extracting the predicted category and confidence, and storing these results along with the original text and true label.

In [ ]:
review_categories = ['positive review', 'negative review']

full_classification_results = []

for i, item in enumerate(processed_data):
    text_content = item['text']
    true_label = item['true_label']

    # Classify the text
    result = classifier(text_content, review_categories)

    # Store the classification details
    classification_entry = {
        "text": text_content,
        "true_label": true_label,
        "predicted_category": result["labels"][0],
        "confidence": round(result["scores"][0], 3)
    }
    full_classification_results.append(classification_entry)

print(f"Performed {len(full_classification_results)} classifications.")
print("First 3 classification results:")
for i in range(min(3, len(full_classification_results))):
    print(full_classification_results[i])

**Reasoning**:
Now that the classifications are complete, I need to calculate the accuracy of the zero-shot model by comparing the predicted categories with the true labels for all 200 samples. This is a direct step towards fulfilling the overall task's accuracy requirement.

In [ ]:
correct_predictions = 0
total_predictions = len(full_classification_results)

for entry in full_classification_results:
    if entry['predicted_category'] == entry['true_label']:
        correct_predictions += 1

accuracy = correct_predictions / total_predictions

print(f"Total samples classified: {total_predictions}")
print(f"Correct predictions: {correct_predictions}")
print(f"Accuracy: {accuracy:.2%}")

**Reasoning**:
To fulfill the requirement of presenting a summary table, I will convert the `full_classification_results` list into a pandas DataFrame. This will allow for a clear, tabular display of the true label, predicted category, and confidence for each review, which is a common and readable format for classification results.

In [ ]:
review_summary_df = pd.DataFrame(full_classification_results)

# Optional: Preview the text content to make the table more readable
review_summary_df['Text Preview'] = review_summary_df['text'].apply(lambda x: x[:100] + '...' if len(x) > 100 else x)

print("\nSummary Table of Classification Results:")
print("=" * 80)
print(review_summary_df[['Text Preview', 'true_label', 'predicted_category', 'confidence']].to_string(index=False))

**Reasoning**:
To visualize the distribution of confidence scores, as required by the main task, I will use a histogram and a Kernel Density Estimate (KDE) plot. This will help in understanding the spread and concentration of the confidence scores from the zero-shot classification. I need to import matplotlib.pyplot and seaborn for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Extract confidence scores from the DataFrame
confidence_scores = review_summary_df['confidence']

plt.figure(figsize=(10, 6))
sns.histplot(confidence_scores, bins=20, kde=True, color='skyblue', edgecolor='black')
plt.title('Distribution of Confidence Scores')
plt.xlabel('Confidence Score')
plt.ylabel('Frequency / Density')
plt.grid(axis='y', alpha=0.75)
plt.show()

### Summary of Findings

**Accuracy of Zero-Shot Classification**: The model achieved an accuracy of **89%** in classifying the 200 Amazon polarity test samples into 'positive review' or 'negative review' categories. This indicates a strong performance for zero-shot classification on this dataset.

**Confidence Score Distribution**: The visualization of confidence scores reveals that a significant portion of the classifications were made with high confidence (scores above 0.7, with a notable peak near 1.0). This suggests that the model was generally certain about its predictions for most of the reviews. There are fewer classifications with low confidence, which is desirable as it implies less ambiguity in the model's predictions.

**Overall Observation**: The zero-shot classification model, using `facebook/bart-large-mnli`, performed effectively on the Amazon polarity dataset. The high accuracy combined with a distribution of confidence scores skewed towards higher values demonstrates its capability to handle sentiment classification without explicit training on domain-specific labeled data.

## Summary:

### Data Analysis Key Findings
*   **Dataset Preparation:** Two hundred samples from the 'amazon\_polarity' test split were successfully loaded and processed. Numerical labels (0, 1) were accurately mapped to descriptive string categories ('negative review', 'positive review').
*   **Classification Accuracy:** The zero-shot classification model achieved an accuracy of 89% (0.89) when classifying the 200 Amazon polarity test samples.
*   **Confidence Score Distribution:** The visualization of confidence scores revealed that a significant portion of classifications were made with high confidence, showing a notable peak near 1.0. This indicates the model was generally certain about its predictions.
*   **Overall Performance:** The zero-shot classification model, using `facebook/bart-large-mnli`, demonstrated strong performance on the Amazon polarity dataset, effectively classifying sentiments without explicit training on domain-specific labeled data.

### Insights or Next Steps
*   The strong performance (89% accuracy with high confidence scores) suggests that zero-shot classification can be a highly effective and efficient method for sentiment analysis tasks, potentially reducing the need for extensive labeled datasets.
*   Further investigation could involve analyzing the 11% of misclassified samples to understand common error patterns or ambiguous cases. This could inform strategies for prompt engineering or identify specific categories where the model struggles, leading to potential improvements.

## A little bit extra: Few-shot examples
Using a different model: **all-MiniLM-L6-v2** is a lightweight sentence-transformer model designed for fast, high-quality sentence embeddings.
It has only 6 Transformer layers (making it small and efficient) but still produces strong semantic similarity and clustering performance, making it ideal for tasks like semantic search, zero-shot clustering, and few-shot classification.

In few-shot classification, we encode a few labeled examples, average them to create class prototypes, and classify new text by picking the class with the most similar embedding.

This approach requires no training, runs locally, and is easy to visualize — making it ideal for quick analytics tasks or demos.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

# few-shot examples
examples = {
    "billing issue": ["charged twice", "overbilling", "incorrect invoice"],
    "technical problem": ["app crash", "screen frozen", "bug in login"],
}

# create prototype embeddings
prototypes = {
    label: model.encode(sentences).mean(axis=0)
    for label, sentences in examples.items()
}

def classify(message):
    emb = model.encode(message)

    # raw cosine similarities (tensor values)
    sims = {label: float(util.cos_sim(emb, proto)) for label, proto in prototypes.items()}

    # convert similarity scores into normalized confidence (softmax)
    scores = np.array(list(sims.values()))
    exp_scores = np.exp(scores)
    confidences = exp_scores / exp_scores.sum()

    # map back to labels
    confidence_dict = {
        label: float(conf)
        for label, conf in zip(sims.keys(), confidences)
    }

    # predicted label = highest confidence
    predicted_label = max(confidence_dict, key=confidence_dict.get)

    return predicted_label, confidence_dict

In [ ]:
classify("I was overcharged $10")

In [ ]:
classify("I was charged 20 when the discount should make it 15, then the app crashed")